# Eksperimen CODAMOSA vs CoverUp

Notebook ini menjalankan **generator CODAMOSA dan CoverUp**, kemudian mengevaluasi
test yang dihasilkan dengan evaluator bersama di `eval_metrics.py` untuk HumanEval
dan Refactory. Default notebook adalah **pilot satu task per dataset**. Ubah
`TASK_LIMIT=None` di `.env` setelah pilot berhasil untuk menjalankan semua task.

Credential hanya dibaca dari environment kernel dan tidak disimpan ke notebook.
Preflight mengirim satu request kecil ke provider, sehingga request tersebut dapat
dikenai biaya.

### Format model

Notebook ini dikonfigurasi untuk endpoint OpenAI-compatible OctaLabs
(`https://router.octalabs.id/v1`). Untuk CoverUp, `openai/` hanyalah prefix
transport LiteLLM; contoh `openai/claude-sonnet-5` dikirim sebagai
`claude-sonnet-5`. CODAMOSA mengirim bare model ID secara langsung.
Alias lama `entah/...` dinormalisasi otomatis. Setiap model dijalankan terpisah.

Jika salah satu metode gagal, notebook tidak mengarang skor. Status `BLOCKED` dan
diagnosis disimpan di `experiment_results/status.json`; metode lain tetap dicoba.


In [1]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import time
import tempfile
import importlib.util

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(dotenv_path, override=False):
        path = Path(dotenv_path)
        if not path.exists():
            return False
        for raw_line in path.read_text(encoding='utf-8').splitlines():
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            key, value = key.strip(), value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in (chr(39), chr(34)):
                value = value[1:-1]
            if override or key not in os.environ:
                os.environ[key] = value
        return True

ROOT = Path.cwd().resolve()
if not (ROOT / 'eval_metrics.py').exists():
    ROOT = Path('/home/adamf/stacks/PRPL').resolve()
if not (ROOT / 'eval_metrics.py').exists():
    raise FileNotFoundError('Buka notebook dari /home/adamf/stacks/PRPL atau sesuaikan ROOT.')
sys.path.insert(0, str(ROOT))
import eval_metrics as em

# Fail early with the actual kernel interpreter instead of producing misleading
# per-task BLOCKED messages when VS Code selected a stale Linux .venv kernel.
REQUIRED_HOST_PACKAGES = ('litellm', 'requests', 'dotenv', 'nbclient')
missing_host_packages = [name for name in REQUIRED_HOST_PACKAGES if importlib.util.find_spec(name) is None]
if missing_host_packages:
    raise RuntimeError(
        'Dependency host tidak lengkap pada interpreter notebook ' + sys.executable
        + ': ' + ', '.join(missing_host_packages)
        + '. Jalankan python -m pip install -r requirements.txt.'
    )
print('Notebook interpreter:', sys.executable)
print('Host dependencies: OK (' + ', '.join(REQUIRED_HOST_PACKAGES) + ')')

load_dotenv(ROOT / '.env', override=True)
if os.getenv('OPENAI_BASE_URL') and not os.getenv('OPENAI_API_BASE'):
    os.environ['OPENAI_API_BASE'] = os.environ['OPENAI_BASE_URL']
os.environ.setdefault('LITELLM_LOCAL_MODEL_COST_MAP', 'True')

DATASETS = ['HumanEval', 'Refactory']
_task_limit_raw = os.getenv('TASK_LIMIT', '1').strip()
TASK_LIMIT = None if _task_limit_raw.lower() in {'', 'none', 'all', '0'} else int(_task_limit_raw)

def provider_model_id(model):
    model = model.strip()
    return model[len('entah/'):] if model.startswith('entah/') else model

def coverup_model_name(model):
    model = provider_model_id(model)
    return model if model.startswith('openai/') else 'openai/' + model

def codamosa_model_name(model):
    model = provider_model_id(model)
    return model[len('openai/'):] if model.startswith('openai/') else model

COVERUP_MODELS = [coverup_model_name(m) for m in os.getenv('COVERUP_MODELS', 'openai/claude-sonnet-5').split(',') if m.strip()]
COVERUP_TIMEOUT = int(os.getenv('COVERUP_TIMEOUT', '900'))
CODAMOSA_MODELS = [codamosa_model_name(m) for m in os.getenv('CODAMOSA_MODELS', 'claude-sonnet-5').split(',') if m.strip()]
assert COVERUP_MODELS and all(m.startswith('openai/') for m in COVERUP_MODELS), COVERUP_MODELS
assert CODAMOSA_MODELS and all(not m.startswith('openai/') for m in CODAMOSA_MODELS), CODAMOSA_MODELS
CODAMOSA_IMAGE = os.getenv('CODAMOSA_IMAGE', 'pynguin-runner:latest')
CODAMOSA_SEARCH_SECONDS = int(os.getenv('CODAMOSA_SEARCH_SECONDS', '60'))
CODAMOSA_POPULATION = int(os.getenv('CODAMOSA_POPULATION', '50'))
OUT = ROOT / 'experiment_results'
OUT.mkdir(parents=True, exist_ok=True)
import experiment_storage as _storage
store = _storage.ResultStore(OUT)
store.begin(os.getenv('RUN_LABEL', 'manual'))
print('Run candidate dir:', store.candidate)
if os.getenv('FRESH', '0').strip().lower() in {'1', 'true', 'yes'}:
    print('FRESH requested; active results are still retained and will be backed up on promote.')
TASKS = {name: em.task_ids(name, TASK_LIMIT) for name in DATASETS}
_task_filter_raw = os.getenv('TASK_FILTER', '').strip()
if _task_filter_raw:
    _wanted = set()
    for _piece in _task_filter_raw.replace(';', ',').split(','):
        _piece = _piece.strip()
        if not _piece:
            continue
        if ':' in _piece:
            _ds, _tid = _piece.split(':', 1)
            _wanted.add((_ds.strip(), _tid.strip()))
        else:
            _wanted.add((None, _piece))
    TASKS = {ds: [t for t in tids if (ds, str(t)) in _wanted or (None, str(t)) in _wanted]
             for ds, tids in TASKS.items()}
    TASKS = {ds: tids for ds, tids in TASKS.items() if tids}
    print('TASK_FILTER:', _task_filter_raw, '->', TASKS)
RESUME = os.getenv('RESUME', '0').strip().lower() in {'1', 'true', 'yes'}

print('Root:', ROOT)
print('Tasks:', TASKS)
print('CoverUp models:', COVERUP_MODELS)
print('CODAMOSA models configured:', CODAMOSA_MODELS)
print('OPENAI_BASE_URL:', os.getenv('OPENAI_BASE_URL'))
print('LITELLM_LOCAL_MODEL_COST_MAP:', os.getenv('LITELLM_LOCAL_MODEL_COST_MAP'))


Notebook interpreter: c:\python314\python.exe
Host dependencies: OK (litellm, requests, dotenv, nbclient)
Run candidate dir: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Tugas 1\PRPL\experiment_results\runs\20261006T102143Z_manual
Root: C:\Users\adamf\Documents\Kuliah\ITS\Semester 1\Topik Dalam Pengujian Perangkat Lunak\Tugas 1\PRPL
Tasks: {'HumanEval': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101',

## Preflight

Preflight memisahkan masalah credential, model, endpoint, dependency, dan Docker.
API key tidak pernah dicetak. Error provider disaring dari key yang diketahui.


In [2]:
def redact(value):
    text = str(value)
    for secret in (os.getenv("OPENAI_API_KEY"), os.getenv("CODAMOSA_API_KEY")):
        if secret:
            text = text.replace(secret, "[REDACTED_API_KEY]")
    return text[:2500]


def explain_error(exc):
    detail = redact(f"{type(exc).__name__}: {exc}")
    lower = detail.lower()
    if "insufficient_quota" in lower or "credit_balance" in lower:
        action = "Key dikenali tetapi saldo/kredit habis. Periksa billing/kuota provider."
    elif "401" in lower or "unauthorized" in lower or "invalid api key" in lower:
        action = "Credential ditolak. Periksa key, project/provider, dan environment kernel."
    elif "403" in lower or "forbidden" in lower:
        action = "Credential tidak memiliki izin untuk model atau endpoint tersebut."
    elif "404" in lower or "not found" in lower or "model_not_found" in lower:
        action = "Model/route tidak ditemukan. Periksa nama model dan URL endpoint."
    elif "429" in lower or "rate limit" in lower:
        action = "Rate limit tercapai. Kurangi request/paralelisme atau tunggu cooldown."
    elif "405" in lower or "method not allowed" in lower:
        action = "Endpoint tidak menerima method/payload ini. Pastikan path chat-completions benar."
    elif "timeout" in lower or "connection" in lower or "network" in lower:
        action = "Endpoint tidak terjangkau/timeout. Periksa jaringan, URL, dan TLS."
    else:
        action = "Periksa detail provider, model, endpoint, credential, dan dependency."
    return detail + "\nLangkah berikutnya: " + action


def coverup_preflight():
    """One cheap call per model; OK only when every model returns a completion."""
    if not os.getenv("OPENAI_API_KEY"):
        return False, "BLOCKED: OPENAI_API_KEY tidak ditemukan di environment kernel.", None
    if not COVERUP_MODELS:
        return False, "BLOCKED: COVERUP_MODELS kosong.", None
    try:
        import litellm
    except ImportError as exc:
        return False, "BLOCKED: paket litellm tidak tersedia.\n" + explain_error(exc), None
    errors, ok_models = [], []
    for model in COVERUP_MODELS:
        try:
            result = litellm.completion(
                model=model,
                messages=[{"role": "user", "content": "Reply with OK."}],
                max_tokens=1,
                timeout=45,
            )
            if getattr(result, "choices", None):
                ok_models.append(model)
            else:
                errors.append(f"{model}: provider merespons tanpa completion choices.")
        except Exception as exc:
            errors.append(f"{model}: " + explain_error(exc))
    if errors:
        return False, "BLOCKED: preflight CoverUp gagal untuk salah satu model.\n" + "\n---\n".join(errors), None
    return True, f"OK: {len(ok_models)}/{len(COVERUP_MODELS)} model CoverUp menghasilkan completion valid.", ok_models[0]


def codamosa_preflight():
    base = os.getenv("CODAMOSA_BASE_URL", "https://router.octalabs.id/v1").strip().rstrip("/")
    path = os.getenv("CODAMOSA_COMPLETIONS_PATH", "/chat/completions").strip()
    default_style = "chat" if "chat/completions" in path else "legacy"
    style = os.getenv("CODAMOSA_API_STYLE", default_style).strip().lower()
    key = os.getenv("CODAMOSA_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not CODAMOSA_MODELS:
        return False, "BLOCKED: CODAMOSA_MODELS kosong.", None
    if not key:
        return False, "BLOCKED: CODAMOSA_API_KEY atau OPENAI_API_KEY tidak ditemukan di environment kernel.", None
    try:
        import requests
    except ImportError as exc:
        return False, "BLOCKED: paket requests tidak tersedia.\n" + explain_error(exc), None
    errors, ok_models = [], []
    for model in CODAMOSA_MODELS:
        try:
            if base == "https://api.openai.com" and style != "chat":
                url = base + "/v1/engines/" + model + "/completions"
                payload = {"prompt": "Reply OK.", "max_tokens": 1, "temperature": 0}
            elif style == "chat":
                url = base + "/" + path.lstrip("/")
                payload = {"model": model, "messages": [{"role": "user", "content": "Reply OK."}], "max_tokens": 1, "temperature": 0}
            else:
                url = base + "/" + path.lstrip("/")
                payload = {"model": model, "prompt": "Reply OK.", "max_tokens": 1, "temperature": 0}
            response = requests.post(url, json=payload, headers={"Authorization": "Bearer " + key, "Content-Type": "application/json"}, timeout=45)
            response.raise_for_status()
            body = response.json()
            choice = (body.get("choices") or [{}])[0]
            if style == "chat":
                valid = (choice.get("message") or {}).get("content") is not None
            else:
                valid = "text" in choice
            if not valid:
                errors.append(f"{model}: response tidak sesuai mode {style}.")
                continue
            ok_models.append(model)
        except Exception as exc:
            errors.append(f"{model}: " + explain_error(exc))
    if errors:
        return False, "BLOCKED: preflight CODAMOSA gagal untuk salah satu model.\n" + "\n---\n".join(errors), None
    return True, f"OK: {len(ok_models)}/{len(CODAMOSA_MODELS)} model CODAMOSA menghasilkan completion valid ({style}).", ok_models[0]


def docker_preflight():
    if not shutil.which("docker"):
        return False, "BLOCKED: Docker CLI tidak ditemukan. Install Docker Engine/CLI."
    try:
        result = subprocess.run(
            ["docker", "info", "--format", "{{.ServerVersion}}"],
            capture_output=True,
            text=True, encoding="utf-8", errors="replace",
            timeout=20,
        )
        if result.returncode:
            return False, (
                "BLOCKED: Docker daemon tidak dapat diakses: "
                + redact((result.stderr or "") + (result.stdout or ""))
                + ". Start daemon dan pastikan user memiliki izin Docker."
            )
        return True, "OK: Docker daemon aktif (" + (result.stdout or "").strip() + ")."
    except Exception as exc:
        return False, "BLOCKED: pemeriksaan Docker gagal.\n" + explain_error(exc)


coverup_ok, coverup_reason, ACTIVE_COVERUP_MODEL = coverup_preflight()
codamosa_ok, codamosa_reason, ACTIVE_CODAMOSA_MODEL = codamosa_preflight()
docker_ok, docker_reason = docker_preflight()
if codamosa_ok and not docker_ok:
    codamosa_ok = False
    codamosa_reason += "\n" + docker_reason

print("CoverUp:", coverup_reason)
print("CODAMOSA:", codamosa_reason)
print("Docker:", docker_reason)
if coverup_ok:
    print("CoverUp active model:", ACTIVE_COVERUP_MODEL)
if codamosa_ok:
    print("CODAMOSA active model:", ACTIVE_CODAMOSA_MODEL)



CoverUp: OK: 1/1 model CoverUp menghasilkan completion valid.
CODAMOSA: OK: 1/1 model CODAMOSA menghasilkan completion valid (chat).
Docker: OK: Docker daemon aktif (29.5.3).
CoverUp active model: openai/claude-sonnet-5
CODAMOSA active model: claude-sonnet-5


In [3]:
def ensure_codamosa_image():
    check = subprocess.run(
        ["docker", "image", "inspect", CODAMOSA_IMAGE],
        capture_output=True,
        text=True, encoding="utf-8", errors="replace",
    )
    if check.returncode == 0:
        return True, "Menggunakan image yang sudah ada: " + CODAMOSA_IMAGE
    command = [
        "docker", "build", "-t", CODAMOSA_IMAGE,
        "-f", str(ROOT / "codamosa" / "docker" / "Dockerfile"),
        str(ROOT / "codamosa"),
    ]
    try:
        result = subprocess.run(
            command, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=1800
        )
        if result.returncode:
            return False, (
                "BLOCKED: build image CODAMOSA gagal (exit "
                + str(result.returncode) + ").\n"
                + redact(((result.stdout or "") + "\n" + (result.stderr or ""))[-5000:])
                + "\nPeriksa koneksi registry, ruang disk, dan Dockerfile."
            )
        return True, "OK: image CODAMOSA berhasil dibangun."
    except Exception as exc:
        return False, "BLOCKED: build image CODAMOSA gagal.\n" + explain_error(exc)


if codamosa_ok:
    image_ok, image_reason = ensure_codamosa_image()
    print(image_reason)
    if not image_ok:
        codamosa_ok = False
        codamosa_reason += "\n" + image_reason
else:
    image_reason = "Dilewati karena preflight CODAMOSA gagal."


Menggunakan image yang sudah ada: pynguin-runner:latest


## Menjalankan generator dan evaluasi

Test yang berhasil saja masuk ke `rows.csv`. Bila generator gagal, task dicatat
sebagai `BLOCKED` dengan alasan dan saran perbaikan di `status.json`.


In [4]:
rows = []
statuses = []
RESUME_KEYS = set()
if RESUME:
    _active_rows, _active_statuses = _storage.ResultStore._read_active(OUT)
    _selected = {(ds, str(t)) for ds, tids in TASKS.items() for t in tids}
    rows = [r for r in _active_rows if (r.get('dataset'), str(r.get('task'))) in _selected]
    statuses = [x for x in _active_statuses
                if x.get('status') == 'OK'
                and (x.get('dataset'), str(x.get('task'))) in _selected]
    for _item in statuses:
        if _item.get('status') == 'OK':
            RESUME_KEYS.add((_item.get('method'), _item.get('model'),
                             _item.get('dataset'), str(_item.get('task'))))
    print(f'RESUME=1: loaded {len(rows)} rows and {len(statuses)} statuses.')


def blocked(method, dataset, task, reason):
    item = {
        "method": method,
        "dataset": dataset,
        "task": str(task),
        "status": "BLOCKED",
        "message": redact(reason),
    }
    statuses.append(item)
    print("\n[" + method + " | " + dataset + " " + str(task) + "] BLOCKED")
    print(item["message"])


def validate_generated_suite(method, layout, tests, work):
    validation_work = work / "validation"
    if validation_work.exists():
        shutil.rmtree(validation_work)
    validation_work.mkdir(parents=True)
    try:
        prepared = em.prepare_suite(layout, tests, validation_work)
        collect_env = dict(os.environ, PYTHONPATH=str(validation_work), NO_COLOR="1", PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
        collect_cmd = [
            em.PY, em.PYTEST, "--collect-only", "-q", "-p", "no:cacheprovider",
            "--no-header", "-o", "addopts=", *[p.name for p in prepared["pytest_files"]]
        ]
        collect = subprocess.run(
            collect_cmd, cwd=str(validation_work), env=collect_env,
            capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=120,
        )
        (work / "pytest_collect.stdout.log").write_text(collect.stdout or "", encoding="utf-8")
        (work / "pytest_collect.stderr.log").write_text(collect.stderr or "", encoding="utf-8")
        if collect.returncode != 0 or not prepared["units"]:
            raise RuntimeError(
                "Suite " + method + " gagal import/pytest collection sebelum evaluasi. Inspect "
                + str(work / "pytest_collect.stderr.log")
                + "\n"
                + redact(((collect.stdout or "") + "\n" + (collect.stderr or ""))[-3000:])
            )
        return prepared
    finally:
        shutil.rmtree(validation_work, ignore_errors=True)


def _model_slug(model):
    return model.replace("/", "_")


def _run_coverup_task_with_model(dataset, task, model):
    layout = em.LAYOUTS[dataset](str(task))
    work = OUT / "work" / "coverup" / _model_slug(model) / dataset / str(task)
    result = em.run_coverup(
        Path(layout["put"]),
        work / "tests",
        work,
        model=model,
        timeout=COVERUP_TIMEOUT,
    )
    if not result.ok:
        raise RuntimeError(
            result.error or "CoverUp exited non-zero; inspect " + str(work / "coverup.log")
        )
    if not result.generated_tests:
        raise RuntimeError(
            "CoverUp selesai tetapi tidak menghasilkan test_coverup_*.py; inspect "
            + str(work / "coverup.log")
        )
    generated = [Path(p) for p in result.generated_tests]
    validate_generated_suite("CoverUp", layout, generated, work)
    row = em.evaluate(
        dataset, str(task), generated, "CoverUp",
        wall_time_s=result.wall_time_s, cost_usd=result.cost_usd,
        workroot=OUT / "work",
    )
    row["generator_ok"] = True
    row["model"] = model
    return row


def _run_codamosa_task_with_model(dataset, task, model):
    layout = em.LAYOUTS[dataset](str(task))
    source = Path(layout["put"]).resolve()
    module = source.stem
    work = OUT / "work" / "codamosa" / _model_slug(model) / dataset / str(task)
    project = work / "project"
    output = work / "output"
    if work.exists():
        shutil.rmtree(work)
    project.mkdir(parents=True)
    output.mkdir(parents=True)
    shutil.copy2(source, project / source.name)
    (project / "package.txt").write_text("")

    base = os.getenv("CODAMOSA_BASE_URL", "https://router.octalabs.id/v1").strip().rstrip("/")
    relative = os.getenv("CODAMOSA_COMPLETIONS_PATH", "/chat/completions").strip()
    key = os.getenv("CODAMOSA_API_KEY") or os.getenv("OPENAI_API_KEY", "")
    args = [
        "--assertion-generation", "MUTATION_ANALYSIS",
        "--project_path", "/input",
        "--module-name", module,
        "--output-path", "/output",
        "--maximum_search_time", str(CODAMOSA_SEARCH_SECONDS),
        "--algorithm", "CODAMOSA",
        "--model_name", model,
        "--model_base_url", base,
        "--model_relative_url", relative,
        "--temperature", "0.8",
        "--large_language_model_seeding", "True",
        "--uninterpreted_statements", "ONLY",
        "--include-partially-parsable", "True",
        "--allow-expandable-cluster", "True",
        "--seeded_testcases_reuse_probability", "1",
        "--population", str(CODAMOSA_POPULATION),
    ]
    auth_file = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", prefix="codamosa-auth-", encoding="utf-8", delete=False
        ) as handle:
            auth_file = Path(handle.name)
            handle.write("--authorization-key\n" + key + "\n")
        auth_file.chmod(0o600)
        command = [
            "docker", "run", "--rm",
            "-v", str(project) + ":/input:ro",
            "-v", str(output) + ":/output",
            "-v", str(project) + ":/package:ro",
            "-v", str(auth_file) + ":/run/codamosa-auth:ro",
            CODAMOSA_IMAGE,
            "@/run/codamosa-auth",
        ] + args
        started = time.perf_counter()
        try:
            result = subprocess.run(
                command,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=CODAMOSA_SEARCH_SECONDS + 600,
            )
        except subprocess.TimeoutExpired as exc:
            raise RuntimeError(
                "CODAMOSA container timeout.\n" + redact((exc.stdout or "") + (exc.stderr or ""))
            ) from exc
    finally:
        if auth_file is not None:
            auth_file.unlink(missing_ok=True)
    elapsed = time.perf_counter() - started
    (work / "docker.stdout.log").write_text(result.stdout or "", encoding="utf-8")
    (work / "docker.stderr.log").write_text(result.stderr or "", encoding="utf-8")
    if result.returncode:
        raise RuntimeError(
            "CODAMOSA/Docker exit code " + str(result.returncode) + ":\n"
            + redact(((result.stdout or "") + "\n" + (result.stderr or ""))[-5000:])
        )
    generated = []
    invalid = []
    for candidate in sorted(output.rglob("*.py")):
        if candidate.name.endswith("_failing.py"):
            continue
        try:
            source_text = candidate.read_text(encoding="utf-8")
            compile(source_text, str(candidate), "exec")
            if em.test_functions(candidate):
                generated.append(candidate)
            else:
                invalid.append(candidate.name + " (no test functions)")
        except (OSError, SyntaxError, UnicodeError) as exc:
            invalid.append(candidate.name + " (" + type(exc).__name__ + ")")
    if not generated:
        # Pynguin can leave the usable suite in *_failing.py while the normal
        # file holds only its header; fall back to it for evaluation.
        for candidate in sorted(output.rglob("*_failing.py")):
            try:
                compile(candidate.read_text(encoding="utf-8"), str(candidate), "exec")
                if em.test_functions(candidate):
                    generated.append(candidate)
                    break
            except (OSError, SyntaxError, UnicodeError):
                continue
    if not generated:
        detail = "Tidak ada suite test CODAMOSA yang valid."
        if invalid:
            detail += " Artefak diabaikan: " + ", ".join(invalid)
        detail += ". Inspect " + str(work / "docker.stdout.log")
        raise RuntimeError(detail)

    validate_generated_suite("CODAMOSA", layout, generated, work)

    row = em.evaluate(
        dataset, str(task), generated, "CODAMOSA",
        wall_time_s=elapsed, workroot=OUT / "work",
    )

    row["generator_ok"] = True
    row["model"] = model
    return row


# Matrix: method x model x dataset x task. Each model has isolated outputs.
def _done(method, model, dataset, task):
    return (method, model, dataset, str(task)) in RESUME_KEYS

for dataset, task_ids in TASKS.items():
    for task in task_ids:
        if coverup_ok:
            for model in COVERUP_MODELS:
                if _done("CoverUp", model, dataset, task):
                    print(f"RESUME skip CoverUp {model} {dataset} {task}")
                    continue
                try:
                    print("CoverUp mencoba model:", model)
                    row = _run_coverup_task_with_model(dataset, task, model)
                    rows.append(row)
                    statuses.append({"method": "CoverUp", "model": model,
                                     "dataset": dataset, "task": str(task),
                                     "status": "OK", "message": "Generation and evaluation completed."})
                    print("CoverUp OK", model, dataset, task)
                except Exception as exc:
                    blocked("CoverUp:" + model, dataset, task, explain_error(exc))
        else:
            for model in COVERUP_MODELS:
                if not _done("CoverUp", model, dataset, task):
                    blocked("CoverUp:" + model, dataset, task, coverup_reason)

        if codamosa_ok:
            for model in CODAMOSA_MODELS:
                if _done("CODAMOSA", model, dataset, task):
                    print(f"RESUME skip CODAMOSA {model} {dataset} {task}")
                    continue
                try:
                    print("CODAMOSA mencoba model:", model)
                    row = _run_codamosa_task_with_model(dataset, task, model)
                    rows.append(row)
                    statuses.append({"method": "CODAMOSA", "model": model,
                                     "dataset": dataset, "task": str(task),
                                     "status": "OK", "message": "Generation and evaluation completed."})
                    print("CODAMOSA OK", model, dataset, task)
                except Exception as exc:
                    blocked("CODAMOSA:" + model, dataset, task, explain_error(exc))
        else:
            for model in CODAMOSA_MODELS:
                if not _done("CODAMOSA", model, dataset, task):
                    blocked("CODAMOSA:" + model, dataset, task, codamosa_reason)

        store.checkpoint(rows, statuses)
        print(f"Checkpoint after {dataset} {task}: rows={len(rows)} statuses={len(statuses)}")


RESUME=1: loaded 4 rows and 4 statuses.
RESUME skip CoverUp openai/claude-sonnet-5 HumanEval 0
RESUME skip CODAMOSA claude-sonnet-5 HumanEval 0
Checkpoint after HumanEval 0: rows=4 statuses=4
CoverUp mencoba model: openai/claude-sonnet-5
CoverUp OK openai/claude-sonnet-5 HumanEval 1
CODAMOSA mencoba model: claude-sonnet-5
CODAMOSA OK claude-sonnet-5 HumanEval 1
Checkpoint after HumanEval 1: rows=6 statuses=6
CoverUp mencoba model: openai/claude-sonnet-5
CoverUp OK openai/claude-sonnet-5 HumanEval 2
CODAMOSA mencoba model: claude-sonnet-5
CODAMOSA OK claude-sonnet-5 HumanEval 2
Checkpoint after HumanEval 2: rows=8 statuses=8
CoverUp mencoba model: openai/claude-sonnet-5
CoverUp OK openai/claude-sonnet-5 HumanEval 3
CODAMOSA mencoba model: claude-sonnet-5
CODAMOSA OK claude-sonnet-5 HumanEval 3
Checkpoint after HumanEval 3: rows=10 statuses=10
CoverUp mencoba model: openai/claude-sonnet-5

[CoverUp:openai/claude-sonnet-5 | HumanEval 4] BLOCKED
RuntimeError: CoverUp selesai tetapi tidak m

KeyboardInterrupt: 

## Simpan hasil

`rows.csv` hanya berisi task yang selesai digenerate dan dievaluasi. `status.json`
berisi status semua task, termasuk alasan metode yang belum dapat berjalan.


In [ ]:
if _task_filter_raw:
    _selected = {(ds, str(t)) for ds, tids in TASKS.items() for t in tids}
    _ar, _ast = _storage.ResultStore._read_active(OUT)
    rows = [r for r in _ar if (r.get('dataset'), str(r.get('task'))) not in _selected] + rows
    statuses = [x for x in _ast if (x.get('dataset'), str(x.get('task'))) not in _selected] + statuses
    print(f'TASK_FILTER merge: {len(rows)} rows and {len(statuses)} statuses after merge.')
expected_statuses = sum(len(task_ids) for task_ids in TASKS.values()) * (len(COVERUP_MODELS) + len(CODAMOSA_MODELS))
if _task_filter_raw:
    expected_statuses = len(statuses)
manifest = store.finalize(rows, statuses, expected_statuses=expected_statuses)
print("Run manifest:", json.dumps(manifest, indent=2, ensure_ascii=False))
if not manifest["promoted"]:
    print("Active results preserved; inspect candidate:", store.candidate)

print("Rows:", OUT / "rows.csv")
print("Statuses:", OUT / "status.json")
print("OK:", sum(item["status"] == "OK" for item in statuses))
print("BLOCKED:", sum(item["status"] == "BLOCKED" for item in statuses))

print("\nResult summary:")
print("OK:", sum(item["status"] == "OK" for item in statuses))
print("BLOCKED:", sum(item["status"] == "BLOCKED" for item in statuses))
if rows:
    summary_groups = {}
    for row in rows:
        key = (row.get("dataset"), row.get("method"))
        summary_groups.setdefault(key, []).append(row)
    for (dataset, method), group in sorted(summary_groups.items()):
        def avg(field):
            vals = [float(r[field]) for r in group if r.get(field) not in (None, "")]
            return round(sum(vals) / len(vals), 2) if vals else None
        print(
            f"{dataset} | {method}: "
            f"tasks={len(group)}, "
            f"coverage={avg('line_pct')}, "
            f"pass_rate={avg('pass_rate')}, "
            f"mutation_score={avg('mutation_score')}, "
            f"wall_time_s={avg('wall_time_s')}, "
            f"cost_usd={avg('cost_usd')}"
        )
else:
    print("No successful rows; inspect status.json for blocked methods/tasks.")

